# elasticsearch smoke test

Tests the `elasticsearch` Python client package with 4 self-contained API-surface examples (no live Elasticsearch cluster required).

In [ ]:
# Example 1: Construct a synchronous Elasticsearch client (no connection attempted).
from elasticsearch import Elasticsearch

client = Elasticsearch(
    hosts=["http://localhost:9200"],
    request_timeout=1,
    max_retries=0,
)

print(f"client type: {type(client).__name__}")
print(f"client transport: {type(client.transport).__name__}")
print(f"has indices namespace: {hasattr(client, 'indices')}")
print(f"has cluster namespace: {hasattr(client, 'cluster')}")
print(f"has search method: {callable(getattr(client, 'search', None))}")

assert type(client).__name__ == "Elasticsearch"
assert hasattr(client, "indices")
assert hasattr(client, "cluster")

In [ ]:
# Example 2: AsyncElasticsearch client construction (no connection attempted).
from elasticsearch import AsyncElasticsearch

aclient = AsyncElasticsearch(
    hosts=["http://localhost:9200"],
    request_timeout=1,
    max_retries=0,
)

print(f"async client type: {type(aclient).__name__}")
print(f"has async search: {callable(getattr(aclient, 'search', None))}")
print(f"has indices namespace: {hasattr(aclient, 'indices')}")

# Close the underlying transport cleanly to avoid resource warnings.
await aclient.close()

assert type(aclient).__name__ == "AsyncElasticsearch"
assert hasattr(aclient, "indices")

In [ ]:
# Example 3: Exception hierarchy — primary public errors are importable and well-formed.
# elasticsearch 9.x re-exports elastic-transport's 9.x hierarchy: ConnectionTimeout
# subclasses TransportError DIRECTLY (no longer a ConnectionError child).
from elasticsearch import (
    ApiError,
    ConflictError,
    ConnectionError,
    ConnectionTimeout,
    NotFoundError,
    RequestError,
    TransportError,
)

assert issubclass(NotFoundError, ApiError)
assert issubclass(ConflictError, ApiError)
assert issubclass(RequestError, ApiError)
assert issubclass(ConnectionError, TransportError)
assert issubclass(ConnectionTimeout, TransportError)
assert not issubclass(ConnectionTimeout, ConnectionError)

print("ApiError subclasses:")
for cls in (NotFoundError, ConflictError, RequestError):
    print(f"  {cls.__name__}: MRO depth {len(cls.__mro__)}")
print("\nTransportError subclasses:")
for cls in (ConnectionError, ConnectionTimeout):
    print(f"  {cls.__name__}: MRO depth {len(cls.__mro__)}")

In [ ]:
# Example 4: Helpers module — bulk/scan/streaming_bulk are importable and have the expected signatures.
import inspect

from elasticsearch import helpers

expected = ["bulk", "streaming_bulk", "parallel_bulk", "scan", "reindex"]
found = {}
for name in expected:
    fn = getattr(helpers, name, None)
    assert callable(fn), f"helpers.{name} is missing or not callable"
    sig = inspect.signature(fn)
    found[name] = list(sig.parameters)[:3]

for name, params in found.items():
    print(f"helpers.{name}({', '.join(params)}, ...)")

# bulk and streaming_bulk must accept (client, actions, ...) as their leading args.
assert found["bulk"][0] == "client"
assert found["streaming_bulk"][0] == "client"